# Orquestador Silver — Transformación completa

Ejecuta los 4 notebooks de transformación Silver en secuencia.  
**Prerequisito:** Bronze debe estar poblado (`run_bronze` completado exitosamente).

| # | Notebook | Tablas Bronze → Silver |
|---|---|---|
| 1 | `stage_mspas` | `dbx_*` (3 tablas) |
| 2 | `stage_ine_edad` | `ine_defunciones_*_edad_*` (3 tablas) |
| 3 | `stage_ine_geografia` | `ine_defunciones_*_depto_*` (2 tablas) |
| 4 | `stage_who` | `who_guatemala`, `who_costa_rica` |

> **Idempotente:** usa `WRITE_MODE = overwrite` — se puede re-ejecutar sin borrar tablas manualmente.  
> **Timeout por notebook:** 1800 s (30 min). Los datos son pequeños; ajusta si es necesario.

In [0]:
import time

# ── Detecta la raíz del repo dinámicamente ────────────────────────────────────
_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
_this_path = _ctx.notebookPath().get()
BASE = "/".join(_this_path.split("/")[:-2])  # sube 2 niveles → raíz del repo

TIMEOUT_SEC = 1800  # 30 min por notebook

_NOTEBOOKS = [
    ("MSPAS",            f"{BASE}/transformation-scripts/stage/stage_mspas"),
    ("INE — Edad",       f"{BASE}/transformation-scripts/stage/stage_ine_edad"),
    ("INE — Geografía",  f"{BASE}/transformation-scripts/stage/stage_ine_geografia"),
    ("WHO",              f"{BASE}/transformation-scripts/stage/stage_who"),
]

print(f"Raíz del repo detectada: {BASE}")
print(f"Notebooks a ejecutar: {len(_NOTEBOOKS)}")

In [0]:
results = []

for label, path in _NOTEBOOKS:
    print(f"\n{'='*60}")
    print(f"▶ Iniciando: {label}")
    print(f"  Path: {path}")
    t0 = time.time()
    try:
        output = dbutils.notebook.run(path, TIMEOUT_SEC)
        elapsed = time.time() - t0
        status = "OK"
        print(f"✓ Completado en {elapsed:.0f}s")
        if output:
            print(f"  Salida: {output}")
    except Exception as e:
        elapsed = time.time() - t0
        status = f"ERROR: {e}"
        print(f"✗ Error en {elapsed:.0f}s: {e}")
    results.append((label, status, f"{elapsed:.0f}s"))

# ── Resumen final ──────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("RESUMEN SILVER")
print(f"{'='*60}")
all_ok = True
for label, status, elapsed in results:
    icon = "✓" if status == "OK" else "✗"
    print(f"  {icon} {label:<20} {elapsed:>6}  {status}")
    if status != "OK":
        all_ok = False

print(f"\nEstado final: {'TODOS OK' if all_ok else 'HAY ERRORES — revisa los notebooks individuales'}")
dbutils.notebook.exit("OK" if all_ok else "ERRORS")